# 04 Compare Petrobras Runs

This notebook flattens one or more run summaries into a comparison CSV and gives a quick ranking view across runs and models.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
import pandas as pd
import plotly.express as px

def find_repo_root() -> Path:
    env_repo = os.getenv('PIPELINE_NOTEBOOK_REPO_ROOT')
    if env_repo:
        return Path(env_repo).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'scripts' / 'compare_petrobras_runs.py').exists():
            return candidate
    return Path('/content/workspace/pipeline-leak-detection')

REPO_ROOT = find_repo_root()
default_storage = Path('/content/drive/MyDrive/pipeline-leak-detection') if Path('/content').exists() else REPO_ROOT / 'artifacts' / 'notebook_runs'
STORAGE_ROOT = Path(os.getenv('PIPELINE_NOTEBOOK_STORAGE_ROOT', str(default_storage))).expanduser().resolve()
DISABLE_PLOTS = os.getenv('PIPELINE_NOTEBOOK_DISABLE_PLOTS', '0') == '1'

ARTIFACTS_ROOT = STORAGE_ROOT / 'artifacts' / 'petrobras'
SUMMARY_GLOB = str(ARTIFACTS_ROOT / '*' / 'models' / 'petrobras_training_summary.json')
COMPARISON_CSV = ARTIFACTS_ROOT / 'petrobras_run_comparison.csv'

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'compare_petrobras_runs.py'),
        '--summary-glob',
        SUMMARY_GLOB,
        '--output-csv',
        str(COMPARISON_CSV),
    ],
    cwd=REPO_ROOT,
    check=True,
)


In [ ]:
comparison = pd.read_csv(COMPARISON_CSV)
comparison.sort_values(['run_label', 'roc_auc', 'f1'], ascending=[True, False, False])


In [ ]:
best_only = comparison[comparison['is_best_model']].copy()
fig = px.bar(best_only, x='run_label', y='roc_auc', color='model', hover_data=['f1', 'accuracy'])
if DISABLE_PLOTS:
    print('Plot rendering skipped for automated run.')
else:
    fig.show()
